# Equipo 11 · Baselines, M1 y M2 (Colab)

Anotación automática de conectores y operadores (Fuentes Rodríguez, 2009) en comentarios de YouTube.

Este notebook compara cuatro sistemas sobre el corpus piloto anotado, con partición por hilo y 5 semillas:

| Sistema | Qué hace |
|---|---|
| B0 | Todo lo que está en el listado, con su primera categoría |
| B1 | La etiqueta más frecuente de cada unidad en entrenamiento |
| M1 | Regresión logística en dos pasos con TF-IDF de la ventana, rasgos de posición y embeddings |
| M2 | M1 + conocimiento lingüístico: restricción por el listado, reglas de contexto, categorías gramaticales (spaCy), respaldo para categorías raras y origen de cada decisión |

**Cómo ejecutarlo**
1. `Entorno de ejecución → Cambiar tipo de entorno → CPU` (no hace falta GPU).
2. Ejecutar `Entorno de ejecución → Ejecutar todas`. La celda de la sección 0 pedirá subir los archivos que falten:
   `referencia_piloto.csv`, `csl_excluidas.csv` y `listado_fuentes2009.csv`.
3. Para la sección 10 (corpus de aplicación, 25 nov), poner `APLICAR_CORPUS = True` y subir el CSV normalizado del corpus.

Además, un detector exploratorio de **candidatas a construcciones semilibres** (CSL) busca patrones como [Dale + SN] o [Como si + oración] para que el equipo las revise. No se evalúa ni entra en el criterio de éxito.

Los resultados quedan en la carpeta `resultados/`. Sólo los archivos de `resultados/publicable/` pueden ir al repositorio: no contienen texto de comentarios (políticas de la API de YouTube).

## 0. Configuración

In [ ]:
USAR_EMBEDDINGS = True     # embeddings de la ventana (sentence-transformers); dejar en True para el hito 2
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
SEMILLAS = [0, 1, 2, 3, 4]
TEST = 0.3                 # proporción de hilos en el test
MAX_ABSTENCION = 0.2       # R3: abstención máxima al fijar el umbral
MIN_EJEMPLOS = 5           # categorías con menos ejemplos en entrenamiento → «revisar categoría»

APLICAR_CORPUS = False     # sección 10: aplicar M2 al corpus de aplicación
CORPUS_APLICACION = "data/comentarios_youtube_normalizados.csv"
MUESTRA_APLICACION = 2000  # comentarios del corpus de aplicación (None = todos; más lento)

In [ ]:
import os, sys, subprocess
os.makedirs("data", exist_ok=True)
paquetes = ["spacy"] + (["sentence-transformers"] if USAR_EMBEDDINGS else [])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm", "-q"], check=False)

necesarios = ["referencia_piloto.csv", "csl_excluidas.csv", "listado_fuentes2009.csv"]
if APLICAR_CORPUS:
    necesarios.append(os.path.basename(CORPUS_APLICACION))
faltan = [f for f in necesarios if not os.path.exists(f"data/{f}")]
if faltan:
    print("Faltan:", faltan)
    try:
        from google.colab import files
        for nombre, contenido in files.upload().items():
            open(f"data/{nombre}", "wb").write(contenido)
    except ImportError:
        print("Copiar los archivos a la carpeta data/ y volver a ejecutar.")
print("Archivos en data/:", sorted(os.listdir("data")))

## 1. Funciones

Tres módulos: `pipeline.py` (candidatos, métricas, B0, B1 y M1), `pipeline_m2.py` (M2) y `pipeline_csl.py` (candidatas a CSL; las reglas están en `REGLAS_CSL`).
Las reglas de contexto están en el diccionario `REGLAS` de `pipeline_m2.py`. Para agregar una regla,
escribirla a partir de la gramática o de las definiciones de Fuentes Rodríguez (2009), nunca mirando
los comentarios del piloto: con cinco semillas, todos quedan alguna vez en el test.

In [ ]:
%%writefile pipeline.py
"""Pipeline del hito 2: candidatos, baselines B0/B1 y clasificador M1.

Entradas (carpeta data/):
- referencia_piloto.csv: comment_id, thread_id, is_reply, text_normalized, anotacion ([unidad::ETIQUETA])
- csl_excluidas.csv: construcciones semilibres que no se evalúan en esta etapa
- listado_fuentes2009.csv: listado de Fuentes Rodríguez (2009)
"""
import re
import collections
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

NO = "NO_DISCURSIVO"
LETRA = r"[\wáéíóúñÁÉÍÓÚÑü]"
ETIQ = re.compile(r"\[([^\[\]]+?)::([^\[\]]+)\]")


# ------------------------------------------------------------------ etiquetas
def etiqueta_listado(macro, cat, sub):
    cat = (cat or "").strip(); sub = (sub or "").strip()
    if macro == "Operador":
        return "OP:" + cat.upper()
    if cat.startswith("Ordenador"):
        return "CON:" + ("INICIO" if sub == "De inicio" else sub.upper())
    if cat.startswith("Reformulativo"):
        return "CON:REFORMULATIVOS"
    return "CON:" + cat.upper()


def normalizar_etiqueta(e):
    e = e.strip().upper()
    if e.startswith("CON:REFORMULATIVO"):
        return "CON:REFORMULATIVOS"
    e = e.replace("CON:ORDENADOR:", "CON:").replace("CON:ORDENADORES:", "CON:")
    return e


def cargar_listado(path):
    L = pd.read_csv(path)
    L["forma"] = L["unidad"].str.replace(r"\s*\d+$", "", regex=True).str.strip().str.lower()
    L["etiqueta"] = [etiqueta_listado(m, c, s) for m, c, s in zip(L.macrofuncion, L.categoria, L.subcategoria)]
    admitidas = L.groupby("forma")["etiqueta"].apply(lambda s: list(dict.fromkeys(s))).to_dict()
    formas = sorted(admitidas, key=len, reverse=True)
    rx = re.compile(rf"(?<!{LETRA})(" + "|".join(re.escape(f) for f in formas) + rf")(?!{LETRA})", re.I)
    return admitidas, rx


# ------------------------------------------------------------------ datos
def leer_anotacion(anot):
    """Texto plano y spans de las etiquetas de referencia."""
    if not isinstance(anot, str) or "::" not in anot:
        return None, []
    plano, spans, pos = "", [], 0
    for m in ETIQ.finditer(anot):
        plano += anot[pos:m.start()]
        s = len(plano); plano += m.group(1)
        spans.append((s, len(plano), m.group(1), normalizar_etiqueta(m.group(2))))
        pos = m.end()
    plano += anot[pos:]
    return plano, spans


def construir_candidatos(ref, rx, csl, devolver_excluidas=False):
    """Una fila por coincidencia del listado, con su etiqueta de referencia.
    Devuelve también las etiquetas de referencia que el listado no detecta.
    Las coincidencias que caen dentro de una construcción semilibre (CSL) anotada se excluyen:
    su función pertenece a la construcción completa, no a la palabra suelta."""
    filas, no_detectadas, excluidas = [], [], []
    for _, r in ref.iterrows():
        texto = r["text_normalized"] if isinstance(r["text_normalized"], str) else ""
        plano, spans = leer_anotacion(r["anotacion"])
        if plano is not None:
            texto = plano
        zonas_csl = [(a, b, u) for a, b, u, _ in spans if (r["comment_id"], u) in csl]
        spans = [sp for sp in spans if (r["comment_id"], sp[2]) not in csl]
        usados = set()
        for m in rx.finditer(texto):
            s, e = m.span()
            csl_que_la_contiene = next((u for a, b, u in zonas_csl if a < e and s < b), None)
            if csl_que_la_contiene is not None:
                excluidas.append({"comment_id": r["comment_id"], "unidad": m.group(0), "csl": csl_que_la_contiene})
                continue
            oro = NO; parte_de_otra = False
            for k, (a, b, u, et) in enumerate(spans):
                if a < e and s < b:
                    if k in usados:          # segunda pieza de una unidad mayor («más» + «encima»)
                        parte_de_otra = True
                    else:
                        oro = et; usados.add(k)
                    break
            if parte_de_otra:
                continue
            izq, der = texto[:s], texto[e:]
            filas.append({
                "comment_id": r["comment_id"], "thread_id": r["thread_id"], "is_reply": bool(r["is_reply"]),
                "unidad": m.group(0).lower(), "inicio": s, "texto": texto,
                "ventana": " ".join(izq.split()[-6:]) + " [[" + m.group(0) + "]] " + " ".join(der.split()[:6]),
                "coma_antes": bool(re.search(r"[,;:]\s*$", izq)), "coma_despues": bool(re.match(r"\s*[,;:]", der)),
                "inicio_comentario": izq.strip() == "",
                "inicio_oracion": bool(re.search(r"(^|[.!?…¿¡\n]\s*)$", izq)),
                "fin_enunciado": bool(re.match(r"\s*([.!?…]|$)", der)),
                "oro": oro,
            })
        for k, (a, b, u, et) in enumerate(spans):
            if k not in usados:
                no_detectadas.append({"comment_id": r["comment_id"], "thread_id": r["thread_id"], "unidad": u, "oro": et})
    if devolver_excluidas:
        return pd.DataFrame(filas), pd.DataFrame(no_detectadas), pd.DataFrame(excluidas)
    return pd.DataFrame(filas), pd.DataFrame(no_detectadas)


# ------------------------------------------------------------------ métricas
def macro_de(e):
    return e.split(":")[0] if e != NO else NO


def metricas(oro, pred, n_no_detectadas):
    """F1 de etiqueta completa, F1 de uso discursivo y F1 macro CON/OP.
    Las etiquetas que el listado no detecta cuentan como falsos negativos."""
    oro, pred = list(oro), list(pred)
    tp = sum(o == p != NO for o, p in zip(oro, pred))
    pp = sum(p != NO for p in pred); gp = sum(o != NO for o in oro) + n_no_detectadas
    f1 = lambda t, a, b: 0.0 if t == 0 else 2 * t / (a + b)
    tpd = sum(o != NO and p != NO for o, p in zip(oro, pred))
    res = {"F1_etiqueta": f1(tp, pp, gp), "F1_uso_discursivo": f1(tpd, pp, gp),
           "precision_etiqueta": tp / pp if pp else 0.0, "cobertura_etiqueta": tp / gp if gp else 0.0}
    f1s = []
    for M in ("CON", "OP"):
        t = sum(macro_de(o) == macro_de(p) == M for o, p in zip(oro, pred))
        a = sum(macro_de(p) == M for p in pred); b = sum(macro_de(o) == M for o in oro)
        f1s.append(f1(t, a, b))
    res["F1_macro_CON_OP"] = float(np.mean(f1s))
    return res


# ------------------------------------------------------------------ baselines
def b0(test, admitidas):
    return [admitidas.get(u, [NO])[0] for u in test["unidad"]]


def b1(train, test):
    freq = train.groupby("unidad")["oro"].agg(lambda s: s.value_counts().index[0]).to_dict()
    global_ = train["oro"].value_counts().index[0]
    return [freq.get(u, global_) for u in test["unidad"]]


# ------------------------------------------------------------------ modelo M1
class M1:
    """Dos pasos: (1) uso discursivo o no; (2) categoría restringida a las admitidas para la unidad.
    Rasgos: TF-IDF de la ventana (palabras y caracteres), unidad y rasgos de posición y puntuación.
    Opcionalmente, embeddings de la ventana (en Colab, con sentence-transformers)."""
    RASGOS = ["coma_antes", "coma_despues", "inicio_comentario", "inicio_oracion", "fin_enunciado", "is_reply"]

    def __init__(self, admitidas, embedder=None, C=1.0, seed=0):
        self.admitidas, self.embedder, self.C, self.seed = admitidas, embedder, C, seed

    def _X(self, df, fit=False):
        if fit:
            self.vw = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
            self.vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
            self.vu = TfidfVectorizer(analyzer=lambda u: [u])
            self.vw.fit(df["ventana"]); self.vc.fit(df["ventana"]); self.vu.fit(df["unidad"])
        partes = [self.vw.transform(df["ventana"]), self.vc.transform(df["ventana"]), self.vu.transform(df["unidad"]),
                  csr_matrix(df[self.RASGOS].astype(float).values)]
        if self.embedder is not None:
            partes.append(csr_matrix(self.embedder(df["ventana"].tolist())))
        return hstack(partes).tocsr()

    def fit(self, df):
        X = self._X(df, fit=True)
        y1 = (df["oro"] != NO).astype(int).values
        self.c1 = LogisticRegression(C=self.C, class_weight="balanced", max_iter=2000, random_state=self.seed).fit(X, y1)
        pos = df["oro"] != NO
        self.vistas = df[pos].groupby("unidad")["oro"].apply(set).to_dict()
        if pos.sum() and df.loc[pos, "oro"].nunique() > 1:
            self.c2 = LogisticRegression(C=self.C, max_iter=2000, random_state=self.seed).fit(X[pos.values], df.loc[pos, "oro"])
        else:
            self.c2 = None; self.unica = df.loc[pos, "oro"].iloc[0] if pos.sum() else NO
        return self

    def prob_discursivo(self, df):
        return self.c1.predict_proba(self._X(df))[:, 1]

    def predict(self, df, margen=0.0):
        X = self._X(df); p = self.c1.predict_proba(X)[:, 1]
        P2 = self.c2.predict_proba(X) if self.c2 is not None else None
        out, abst = [], []
        for i, u in enumerate(df["unidad"]):
            if abs(p[i] - 0.5) < margen:
                out.append(NO); abst.append(True); continue
            abst.append(False)
            if p[i] < 0.5:
                out.append(NO); continue
            permitidas = set(self.admitidas.get(u, [])) | self.vistas.get(u, set())
            if P2 is None:
                out.append(self.unica); continue
            clases = list(self.c2.classes_)
            orden = np.argsort(-P2[i])
            elegida = next((clases[j] for j in orden if clases[j] in permitidas), clases[orden[0]])
            out.append(elegida)
        return out, np.array(abst), p


def elegir_margen(train, admitidas, embedder=None, max_abst=0.2, seed=0):
    """Umbral de abstención fijado por validación cruzada agrupada por hilo, solo con entrenamiento."""
    gkf = GroupKFold(n_splits=5)
    probs = np.zeros(len(train))
    for a, b in gkf.split(train, groups=train["thread_id"]):
        m = M1(admitidas, embedder, seed=seed).fit(train.iloc[a])
        probs[b] = m.prob_discursivo(train.iloc[b])
    y = (train["oro"] != NO).values
    mejor = (0.0, -1)
    for mg in np.arange(0, 0.31, 0.05):
        keep = np.abs(probs - 0.5) >= mg
        if 1 - keep.mean() > max_abst or keep.sum() == 0:
            continue
        pr = probs[keep] >= 0.5; yy = y[keep]
        tp = (pr & yy).sum(); f1 = 0 if tp == 0 else 2 * tp / (pr.sum() + yy.sum())
        if f1 > mejor[1] + 1e-9:
            mejor = (round(float(mg), 2), f1)
    return mejor[0]


def bootstrap_dif(test, pred_a, pred_b, n_nd_por_hilo, n=1000, seed=0):
    """IC 95% de la diferencia de F1 de etiqueta (a - b), remuestreando hilos del test."""
    rng = np.random.default_rng(seed)
    hilos = test["thread_id"].unique()
    idx_por_hilo = {h: np.where(test["thread_id"].values == h)[0] for h in hilos}
    difs = []
    for _ in range(n):
        muestra = rng.choice(hilos, size=len(hilos), replace=True)
        idx = np.concatenate([idx_por_hilo[h] for h in muestra])
        nd = sum(n_nd_por_hilo.get(h, 0) for h in muestra)
        o = test["oro"].values[idx]
        fa = metricas(o, np.array(pred_a)[idx], nd)["F1_etiqueta"]
        fb = metricas(o, np.array(pred_b)[idx], nd)["F1_etiqueta"]
        difs.append(fa - fb)
    return np.percentile(difs, [2.5, 97.5])

In [ ]:
%%writefile pipeline_m2.py
"""M2: M1 + conocimiento lingüístico (restricción por el listado, reglas de contexto,
categorías gramaticales de spaCy, respaldo para categorías raras y origen de cada decisión).

Las reglas se escriben a partir de la gramática y de las definiciones de Fuentes Rodríguez (2009),
no mirando los comentarios del piloto: con cinco semillas, todos quedan alguna vez en el test.
"""
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

from pipeline import NO, M1

MIN_EJEMPLOS = 5          # categorías con menos ejemplos en entrenamiento → «revisar categoría»


# ------------------------------------------------------------------ categorías gramaticales
def anotar_gramatica(cand, nlp):
    """Agrega la categoría gramatical (UPOS de spaCy) de la unidad, de la palabra anterior y de la siguiente."""
    cache = {}
    filas = []
    for texto, ini, u in zip(cand["texto"], cand["inicio"], cand["unidad"]):
        if texto not in cache:
            cache[texto] = nlp(texto)
        doc = cache[texto]
        fin = ini + len(u)
        dentro = [t for t in doc if t.idx < fin and t.idx + len(t) > ini]
        antes = [t for t in doc if t.idx + len(t) <= ini]
        despues = [t for t in doc if t.idx >= fin]
        filas.append({
            "pos_u": dentro[0].pos_ if dentro else "X",
            "pos_ant": antes[-1].pos_ if antes else "INICIO",
            "pos_sig": despues[0].pos_ if despues else "FIN",
            "tok_ant": antes[-1].text.lower() if antes else "",
            "tok_sig": despues[0].text.lower() if despues else "",
        })
    return pd.concat([cand.reset_index(drop=True), pd.DataFrame(filas)], axis=1)


# ------------------------------------------------------------------ reglas de contexto
def _pregunta(texto, ini):
    """La unidad está dentro de una oración interrogativa."""
    resto = texto[ini:]
    m = re.search(r"[.!?…\n]", resto)
    return bool(m and m.group(0) == "?") or "¿" in texto[max(0, texto.rfind(".", 0, ini)):ini]


REGLAS = {
    # generales
    "sig_det_o_num": lambda r: r.pos_sig in ("DET", "NUM") or bool(re.match(r"\d", r.tok_sig)),
    "sig_verbo": lambda r: r.pos_sig in ("VERB", "AUX"),
    "ant_verbo": lambda r: r.pos_ant in ("VERB", "AUX"),
    "sig_adj_o_adv": lambda r: r.pos_sig in ("ADJ", "ADV"),
    "ant_det": lambda r: r.pos_ant == "DET",
    "ant_prep": lambda r: r.pos_ant == "ADP",
    "sig_que": lambda r: r.tok_sig == "que",
    "aislada_por_comas": lambda r: r.coma_despues and (r.inicio_oracion or r.coma_antes),
    "fin_tras_verbo": lambda r: r.pos_ant in ("VERB", "AUX") and r.fin_enunciado,
    "en_pregunta": lambda r: _pregunta(r.texto, r.inicio),
    # específicas (ejemplos de la propuesta)
    "hasta_preposicion": lambda r: r.unidad == "hasta" and (r.pos_sig in ("DET", "NUM", "PROPN", "NOUN") or bool(re.match(r"\d", r.tok_sig))),
    "ya_temporal": lambda r: r.unidad == "ya" and not r.coma_despues and r.pos_sig in ("VERB", "AUX", "PRON"),
    "ya_operador": lambda r: r.unidad == "ya" and r.inicio_oracion and r.coma_despues,
    "solo_adjetivo": lambda r: r.unidad == "solo" and r.pos_ant in ("VERB", "AUX", "PRON") and r.fin_enunciado,
    "mas_comparativo": lambda r: r.unidad == "más" and (r.pos_sig in ("ADJ", "ADV", "NOUN", "NUM") or r.tok_sig == "que"),
}


def aplicar_reglas(df):
    return pd.DataFrame({n: [bool(f(r)) for r in df.itertuples()] for n, f in REGLAS.items()}, index=df.index)


# ------------------------------------------------------------------ modelo M2
class M2(M1):
    """M1 + reglas de contexto y categorías gramaticales como rasgos.
    La categoría se restringe estrictamente al listado: si la unidad admite una sola, la fija el listado."""

    def _X(self, df, fit=False):
        base = super()._X(df, fit)
        R = aplicar_reglas(df).astype(float).values
        gram = [{"u=" + a, "ant=" + b, "sig=" + c, f"{u}|sig={c}", f"{u}|ant={b}"}
                for u, a, b, c in zip(df["unidad"], df["pos_u"], df["pos_ant"], df["pos_sig"])]
        gram = [{k: 1 for k in g} for g in gram]
        if fit:
            self.dv = DictVectorizer().fit(gram)
        return hstack([base, csr_matrix(R), self.dv.transform(gram)]).tocsr()

    def fit(self, df):
        super().fit(df)
        self.n_cat = df.loc[df["oro"] != NO, "oro"].value_counts().to_dict()
        return self

    def predict(self, df, margen=0.0, detalle=False):
        X = self._X(df); p = self.c1.predict_proba(X)[:, 1]
        P2 = self.c2.predict_proba(X) if self.c2 is not None else None
        clases = list(self.c2.classes_) if self.c2 is not None else []
        out, abst, origen, revisar = [], [], [], []
        for i, u in enumerate(df["unidad"]):
            adm = self.admitidas.get(u, [])
            if abs(p[i] - 0.5) < margen:
                out.append(NO); abst.append(True); origen.append("modelo"); revisar.append(False); continue
            abst.append(False)
            if p[i] < 0.5:
                out.append(NO); origen.append("modelo"); revisar.append(False); continue
            if len(adm) == 1:
                cat, org = adm[0], "listado"
            elif P2 is not None:
                orden = [clases[j] for j in np.argsort(-P2[i]) if clases[j] in adm]
                cat, org = (orden[0], "modelo") if orden else (adm[0], "listado")
            else:
                cat, org = adm[0], "listado"
            out.append(cat); origen.append(org)
            revisar.append(self.n_cat.get(cat, 0) < MIN_EJEMPLOS)
        if detalle:
            return out, np.array(abst), p, origen, np.array(revisar)
        return out, np.array(abst), p


def elegir_margen_m(cls, train, admitidas, embedder=None, max_abst=0.2, seed=0):
    """Igual que elegir_margen, pero para cualquier clase de modelo (M1 o M2)."""
    gkf = GroupKFold(n_splits=5)
    probs = np.zeros(len(train))
    for a, b in gkf.split(train, groups=train["thread_id"]):
        m = cls(admitidas, embedder, seed=seed).fit(train.iloc[a])
        probs[b] = m.prob_discursivo(train.iloc[b])
    y = (train["oro"] != NO).values
    mejor = (0.0, -1)
    for mg in np.arange(0, 0.31, 0.05):
        keep = np.abs(probs - 0.5) >= mg
        if 1 - keep.mean() > max_abst or keep.sum() == 0:
            continue
        pr = probs[keep] >= 0.5; yy = y[keep]
        tp = (pr & yy).sum(); f1 = 0 if tp == 0 else 2 * tp / (pr.sum() + yy.sum())
        if f1 > mejor[1] + 1e-9:
            mejor = (round(float(mg), 2), f1)
    return mejor[0]

In [ ]:
%%writefile pipeline_csl.py
"""Detector exploratorio de candidatas a construcciones semilibres (CSL).

Cada regla tiene una parte fija (expresión regular) y un hueco que se comprueba con la categoría
gramatical de spaCy. Las reglas salen de las 16 CSL anotadas en el piloto: el detector propone
candidatas para revisión humana; no se evalúa ni entra en el criterio de éxito.

Para agregar una CSL nueva, agregar una entrada a REGLAS_CSL.
"""
import re
import pandas as pd

PERSONALES = {"tú", "tu", "usted", "ustedes", "él", "ella", "ellos", "ellas", "vos", "uno"}
SN = ("PROPN", "NOUN", "PRON", "DET", "ADJ")


def _siguientes(doc, pos, n=6):
    """Hasta n palabras (sin puntuación) que empiezan en pos o después."""
    return [t for t in doc if t.idx >= pos and not t.is_punct and not t.is_space][:n]


def hueco_sn(doc, m):
    """El hueco es un SN; se excluye «gracias a(l)…», que es una locución causal."""
    sig = _siguientes(doc, m.end(), 1)
    return bool(sig) and sig[0].pos_ in SN and sig[0].text.lower() not in ("a", "al")


def hueco_y_sn(doc, m):
    """[Y + SN]: el SN es un pronombre personal o contiene un nombre propio en las 3 palabras siguientes."""
    sig = _siguientes(doc, m.end(), 3)
    if not sig:
        return False
    if sig[0].text.lower() in PERSONALES:
        return True
    return any(t.pos_ == "PROPN" or t.text.lower() == "san" for t in sig) and sig[0].pos_ in SN


def hueco_oracion(doc, m):
    """El hueco es una oración: hay un verbo en las 6 palabras siguientes."""
    return any(t.pos_ in ("VERB", "AUX") for t in _siguientes(doc, m.end(), 6))


def hueco_sustantivo(doc, m):
    sig = _siguientes(doc, m.end(), 1)
    return bool(sig) and sig[0].pos_ in ("NOUN", "PROPN")


def verbo_en_grupo1(doc, m):
    return any(t.idx == m.start(1) and t.pos_ in ("VERB", "AUX") for t in doc)


def hueco_cualquiera(doc, m):
    return bool(_siguientes(doc, m.end(), 1))


INI = r"(?:^|(?<=[.!?…¿¡\n])\s*)"   # inicio de comentario u oración

# nombre, expresión de la parte fija, comprobación del hueco, etiqueta usada en el piloto
REGLAS_CSL = [
    ("[Dale + SN]", r"\bdale\b,?\s*", hueco_sn, "OP:MODAL"),
    ("[Gracias, + SN]", r"\bgracias\b,?\s*", hueco_sn, "OP:MODAL"),
    ("[Y + SN]", INI + r"\by\b\s+", hueco_y_sn, "OP:MODAL"),
    ("[No, si + oración]", r"\bno,\s*si\b", hueco_cualquiera, "OP:MODAL"),
    ("[Ya es + cuantificador]", r"\bya es (?:mucho|demasiado|suficiente|el colmo)\b", None, "OP:MODAL"),
    ("[No + V + nada con + X]", r"\bno\s+(\w+)\s+nada\s+con\b", verbo_en_grupo1, "OP:MODAL"),
    ("[Ni un solo + N]", r"\bni\s+una?\s+sol[oa]\s+", hueco_sustantivo, "OP:ARGUMENTATIVO"),
    ("[Es obvio que + oración]", r"\bes\s+(?:obvio|claro|evidente|l[óo]gico)\s+que\b", None, "OP:MODAL"),
    ("[Por culpa de + SN]", r"\bpor culpa del?\b", hueco_cualquiera, "OP:MODAL"),
    ("[Como si nada]", r"\bcomo si nada\b", None, "OP:MODAL"),
    ("[Como si + oración]", r"\bcomo si\b(?!\s+nada\b)", hueco_oracion, "OP:MODAL"),
    ("[Para qué + decir]", r"\bpara\s+qu[ée]\s+(?:decir|hablar|hablemos)\b", None, "OP:MODAL"),
]
_RX = [(n, re.compile(p, re.I), f, e) for n, p, f, e in REGLAS_CSL]


def detectar_csl(df, nlp, col_texto="text_normalized"):
    """Una fila por candidata a CSL: comentario, regla, posición, coincidencia y contexto."""
    filas = []
    for cid, texto in zip(df["comment_id"], df[col_texto]):
        if not isinstance(texto, str) or not texto.strip():
            continue
        doc = None
        for nombre, rx, hueco, etiqueta in _RX:
            for m in rx.finditer(texto):
                if hueco is not None:
                    doc = doc or nlp(texto)
                    if not hueco(doc, m):
                        continue
                fin = m.end()
                if hueco is not None:
                    sig = _siguientes(doc, m.end(), 3)
                    fin = sig[-1].idx + len(sig[-1]) if sig else m.end()
                filas.append({"comment_id": cid, "regla_csl": nombre, "inicio": m.start(), "fin": fin,
                              "coincidencia": texto[m.start():fin],
                              "contexto": texto[max(0, m.start() - 40):fin + 40].replace("\n", " "),
                              "etiqueta_en_piloto": etiqueta})
    return pd.DataFrame(filas, columns=["comment_id", "regla_csl", "inicio", "fin", "coincidencia",
                                        "contexto", "etiqueta_en_piloto"])


def marcar_dentro_de_csl(cand, csl_det):
    """Marca los candidatos del listado que caen dentro de una candidata a CSL."""
    zonas = csl_det.groupby("comment_id")[["inicio", "fin", "regla_csl"]].apply(lambda g: list(g.itertuples(index=False))).to_dict()
    marca = []
    for cid, s, u in zip(cand["comment_id"], cand["inicio"], cand["unidad"]):
        e = s + len(u)
        z = next((r.regla_csl for r in zonas.get(cid, []) if r.inicio < e and s < r.fin), "")
        marca.append(z)
    return cand.assign(dentro_de_csl=marca)

In [ ]:
import importlib, time, hashlib, json
import numpy as np, pandas as pd
import pipeline, pipeline_m2, pipeline_csl
importlib.reload(pipeline); importlib.reload(pipeline_m2); importlib.reload(pipeline_csl)
from pipeline import *
from pipeline_m2 import *
from pipeline_csl import REGLAS_CSL, detectar_csl, marcar_dentro_de_csl
pipeline_m2.MIN_EJEMPLOS = MIN_EJEMPLOS
from sklearn.model_selection import GroupShuffleSplit
pd.set_option("display.max_colwidth", 120)

## 2. Candidatos y etiquetas de referencia

Cada coincidencia de una unidad del listado es un candidato. Su etiqueta de referencia es la del equipo o `NO_DISCURSIVO`.
Las construcciones semilibres no se evalúan, y tampoco las palabras del listado que caen dentro de una CSL anotada (por ejemplo, 'ya' en «YA ES MUCHO»): su función pertenece a la construcción completa. Las etiquetas que el listado no encuentra cuentan como falsos negativos para todos los sistemas.

In [ ]:
admitidas, rx = cargar_listado("data/listado_fuentes2009.csv")
ref = pd.read_csv("data/referencia_piloto.csv")
csl = set(map(tuple, pd.read_csv("data/csl_excluidas.csv").values))
cand, nd, dentro_csl = construir_candidatos(ref, rx, csl, devolver_excluidas=True)
print(f"Excluidas por estar dentro de una CSL anotada: {len(dentro_csl)}")
print(dentro_csl.groupby("csl").unidad.apply(", ".join).to_string())
print(f"Comentarios: {len(ref)} · hilos: {ref.thread_id.nunique()}")
print(f"Candidatos: {len(cand)} · con uso discursivo: {(cand.oro != NO).sum()} · no detectables por el listado: {len(nd)}")
pd.concat([cand[cand.oro != NO].oro, nd.oro]).value_counts().rename("ejemplos por categoría").to_frame()

### 2.1. Cuánto decide el listado

Si una unidad admite una sola categoría en el listado, M2 sólo decide si hay uso discursivo; la categoría la pone el listado.
La tabla muestra cuántas etiquetas del piloto son de ese tipo y en cuántas el listado acierta. Las que no coinciden son errores asegurados de M2, y se declaran.

In [ ]:
mono = {u for u, v in admitidas.items() if len(v) == 1}
print(f"Formas del listado: {len(admitidas)} · con una sola categoría: {len(mono)}")
pos = cand[cand.oro != NO].copy()
pos["una_categoria"] = pos.unidad.isin(mono)
pos["listado_acierta"] = [u in mono and admitidas[u][0] == o for u, o in zip(pos.unidad, pos.oro)]
n1 = pos.una_categoria.sum(); n2 = pos.listado_acierta.sum()
print(f"Etiquetas detectables: {len(pos)} · de unidades con una sola categoría: {n1} · el listado acierta en {n2} ({n2/max(n1,1):.0%})")
pos[pos.una_categoria & ~pos.listado_acierta][["unidad", "oro", "ventana"]].assign(listado=lambda d: [admitidas[u][0] for u in d.unidad])

## 3. Categorías gramaticales (spaCy) y reglas de contexto

In [ ]:
import spacy
nlp = spacy.load("es_core_news_sm")
cand = anotar_gramatica(cand, nlp)
reglas = aplicar_reglas(cand)
pd.DataFrame({
    "se activa en": reglas.sum(),
    "de ellas, con uso discursivo": reglas[cand.oro != NO].sum(),
}).sort_values("se activa en", ascending=False)

### 3.1. Candidatas a construcciones semilibres (exploratorio)

Las reglas de `REGLAS_CSL` tienen una parte fija y un hueco que se comprueba con spaCy: por ejemplo, [Dale + SN] exige que después de «Dale» venga un sustantivo, nombre propio o pronombre.
Como las reglas salen de las 16 CSL del piloto, encontrarlas aquí no es una evaluación: sólo comprueba que las reglas funcionan.
Lo útil son las coincidencias nuevas, que el equipo debe revisar. Para agregar una CSL, agregar una línea a `REGLAS_CSL`.

In [ ]:
ref["texto_plano"] = [leer_anotacion(a)[0] or t for a, t in zip(ref.anotacion, ref.text_normalized)]
csl_piloto = detectar_csl(ref, nlp, "texto_plano")
anotadas = pd.read_csv("data/csl_excluidas.csv")
encontradas = sum(
    any(u.lower().split()[0] in c.lower() for c in csl_piloto[csl_piloto.comment_id == cid].coincidencia)
    for cid, u in zip(anotadas.comment_id, anotadas.unidad))
print(f"CSL anotadas en el piloto que las reglas encuentran: {encontradas} de {len(anotadas)}")
print(f"Candidatas a CSL en el piloto: {len(csl_piloto)} · por regla:")
print(csl_piloto.regla_csl.value_counts().to_string())
csl_piloto[["regla_csl", "coincidencia", "contexto"]]

## 4. Embeddings

In [ ]:
embedder = None
if USAR_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(MODELO_EMB, device="cpu")
    _cache = {}
    def embedder(textos):
        nuevos = [t for t in dict.fromkeys(textos) if t not in _cache]
        if nuevos:
            for t, v in zip(nuevos, st.encode(nuevos, batch_size=64, show_progress_bar=False)):
                _cache[t] = v
        return np.vstack([_cache[t] for t in textos])
    print("Embeddings:", MODELO_EMB, "·", st.get_sentence_embedding_dimension(), "dimensiones")
else:
    print("ATENCIÓN: sin embeddings. Para el hito 2, USAR_EMBEDDINGS debe ser True.")

## 5. Test congelado

Para cada semilla se separa el 30 % de los hilos como test. Los IDs de cada test se guardan con su hash SHA-256:
si alguien vuelve a ejecutar el notebook y obtiene el mismo hash, evaluó exactamente con los mismos comentarios.

In [ ]:
os.makedirs("resultados/publicable/test_congelado", exist_ok=True)
particiones, hashes = {}, {}
for seed in SEMILLAS:
    tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=TEST, random_state=seed).split(cand, groups=cand.thread_id))
    particiones[seed] = (tr_i, te_i)
    ids = sorted(cand.iloc[te_i].comment_id.unique())
    open(f"resultados/publicable/test_congelado/test_semilla{seed}.txt", "w").write("\n".join(ids))
    hashes[seed] = hashlib.sha256("\n".join(ids).encode()).hexdigest()
    print(f"Semilla {seed}: {len(te_i)} candidatos de test · {len(ids)} comentarios · SHA-256 {hashes[seed][:16]}…")
json.dump(hashes, open("resultados/publicable/test_congelado/hashes.json", "w"), indent=1)

## 6. Experimento: B0, B1, M1 y M2

Para M1 y M2, el umbral de abstención se fija por validación cruzada agrupada por hilo (5 pliegues) **sólo con el entrenamiento** (R3).
Se informan dos versiones: sin abstención (todos los candidatos) y con abstención (sólo lo no abstenido).

In [ ]:
filas, preds = [], {}
for seed in SEMILLAS:
    tr_i, te_i = particiones[seed]
    tr, te = cand.iloc[tr_i].reset_index(drop=True), cand.iloc[te_i].reset_index(drop=True)
    nd_te = nd[nd.thread_id.isin(set(te.thread_id))]
    p0, p1 = b0(te, admitidas), b1(tr, te)
    m1 = M1(admitidas, embedder, seed=seed).fit(tr)
    pm1, _, prob1 = m1.predict(te, 0.0)
    mg1 = elegir_margen_m(M1, tr, admitidas, embedder, MAX_ABSTENCION, seed)
    pm1a, ab1, _ = m1.predict(te, mg1)
    m2 = M2(admitidas, embedder, seed=seed).fit(tr)
    pm2, _, prob2, origen, revisar = m2.predict(te, 0.0, detalle=True)
    mg2 = elegir_margen_m(M2, tr, admitidas, embedder, MAX_ABSTENCION, seed)
    pm2a, ab2, _ = m2.predict(te, mg2)
    for nombre, p in (("B0", p0), ("B1", p1), ("M1", pm1), ("M2", pm2)):
        r = metricas(te.oro, p, len(nd_te)); r.update(sistema=nombre, semilla=seed); filas.append(r)
    for nombre, p, ab, mg in (("M1 con abstención", pm1a, ab1, mg1), ("M2 con abstención", pm2a, ab2, mg2)):
        r = metricas(te.oro[~ab], np.array(p)[~ab], len(nd_te))
        r.update(sistema=nombre, semilla=seed, abstencion=ab.mean(), margen=mg); filas.append(r)
    preds[seed] = dict(
        te=te.assign(B0=p0, B1=p1, M1=pm1, M2=pm2, p_M1=prob1, p_M2=prob2, origen_M2=origen,
                     revisar_categoria=revisar, abstiene_M2=ab2),
        nd_hilo=nd_te.groupby("thread_id").size().to_dict(),
        n_cat_train=m2.n_cat)
    print(f"Semilla {seed} lista")
res = pd.DataFrame(filas)
cols = ["F1_etiqueta", "F1_uso_discursivo", "F1_macro_CON_OP", "precision_etiqueta", "cobertura_etiqueta"]
tabla = res.groupby("sistema", sort=False)[cols].agg(["mean", "std"]).round(3)
tabla

In [ ]:
print("Abstención (R3: ≤ 0,20) y margen elegido por validación cruzada, por semilla:")
res[res.sistema.str.contains("abstención")][["sistema", "semilla", "margen", "abstencion", "F1_etiqueta", "F1_uso_discursivo"]].round(3)

## 7. Criterio de éxito e intervalos de confianza

Criterio: M2 supera a B1 en al menos 0,10 de F1 de etiqueta (promedio de 5 semillas) y el IC 95 % de la diferencia
(bootstrap por hilo, 1.000 repeticiones) no incluye el cero. También se calcula M2 − M1: muestra cuánto aporta el conocimiento lingüístico.

In [ ]:
ics = []
for seed in SEMILLAS:
    te, ndh = preds[seed]["te"], preds[seed]["nd_hilo"]
    for a, b in (("M2", "B1"), ("M2", "M1"), ("M1", "B1")):
        lo, hi = bootstrap_dif(te, te[a].tolist(), te[b].tolist(), ndh, seed=seed)
        ics.append(dict(comparacion=f"{a} − {b}", semilla=seed, IC_inf=lo, IC_sup=hi, excluye_cero=lo > 0 or hi < 0))
ics = pd.DataFrame(ics)
media = res.groupby("sistema").F1_etiqueta.mean()
dif = media["M2"] - media["B1"]
print(f"Diferencia media M2 − B1: {dif:+.3f} (meta ≥ +0,10) → {'cumple' if dif >= 0.10 else 'no cumple'}")
print(f"Diferencia media M2 − M1: {media['M2'] - media['M1']:+.3f}")
ics.round(3)

## 8. Trazabilidad (R7) y categorías raras

Cada etiqueta de M2 indica si la categoría la fijó el listado o la eligió el modelo. Las reglas no deciden: son rasgos,
y la columna `reglas_activas` muestra cuáles se activaron en cada candidato.
La última tabla es la lista para la diapositiva del riesgo: categorías con menos de 5 ejemplos en el test.

In [ ]:
todas = pd.concat([preds[s]["te"].assign(semilla=s) for s in SEMILLAS])
disc = todas[todas.M2 != NO]
print(f"R7: predicciones con origen completo: {todas.origen_M2.notna().mean():.0%}")
print("Origen de las etiquetas con uso discursivo (todas las semillas):", disc.origen_M2.value_counts().to_dict())
print(f"Marcadas «revisar categoría»: {disc.revisar_categoria.sum()} de {len(disc)} ({disc.revisar_categoria.mean():.0%})")
conteo = pd.DataFrame({f"semilla {s}": preds[s]["te"].query("oro != @NO").oro.value_counts() for s in SEMILLAS}).fillna(0).astype(int)
conteo["menos de 5 en todas"] = (conteo < MIN_EJEMPLOS).all(axis=1)
conteo.sort_values("semilla 0", ascending=False)

## 9. Errores (semilla 0) y requisitos operativos

Prioridad para «ya», «hasta» y «solo». Para el error más informativo, mirar qué reglas se activaron y qué categorías gramaticales vio el modelo.

In [ ]:
te0 = preds[SEMILLAS[0]]["te"]
R0 = aplicar_reglas(te0)
te0["reglas_activas"] = [", ".join(R0.columns[R0.loc[i]]) for i in R0.index]
err = te0[te0.M2 != te0.oro].copy()
err["prioridad"] = err.unidad.isin(["ya", "hasta", "solo"])
err.sort_values(["prioridad", "unidad"], ascending=[False, True])[
    ["unidad", "oro", "M1", "M2", "p_M2", "origen_M2", "pos_ant", "pos_u", "pos_sig", "reglas_activas", "ventana"]]

In [ ]:
# R1: detección del 100 % de las formas del listado
prueba = " | ".join(admitidas)
detectadas = {m.group(0).lower() for m in rx.finditer(prueba)}
print(f"R1: formas detectadas {len(detectadas & set(admitidas))} de {len(admitidas)}")

# R5: ningún nombre de usuario en los textos
menciones = cand.texto.str.findall(r"@(?!USER\b)\w+").str.len().sum()
print(f"R5: menciones @nombre sin anonimizar: {menciones}")

# R6: latencia por comentario (spaCy + embeddings + M2), p95 en CPU
m_final = M2(admitidas, embedder, seed=0).fit(cand.iloc[particiones[SEMILLAS[0]][0]].reset_index(drop=True))
tiempos = []
for cid, g in te0.groupby("comment_id"):
    t0 = time.perf_counter()
    g2 = anotar_gramatica(g[[c for c in cand.columns if c not in ("pos_u", "pos_ant", "pos_sig", "tok_ant", "tok_sig")]], nlp)
    if embedder is not None:
        _cache.clear()
    m_final.predict(g2)
    tiempos.append(time.perf_counter() - t0)
print(f"R6: latencia p95 = {np.percentile(tiempos, 95):.3f} s por comentario (meta ≤ 1 s) · mediana {np.median(tiempos):.3f} s")

## 10. Corpus de aplicación (25 nov)

Como el corpus de aplicación no tiene etiquetas, no se puede calcular su F1. Se compara, en cambio, la abstención y la
distribución de categorías predichas con las del test del piloto: si M2 se abstiene mucho más en el corpus nuevo, el cambio de tema
lo está afectando. El modelo se entrena con todo el piloto y el umbral se fija por validación cruzada.
También se buscan candidatas a CSL con las mismas reglas de la sección 3.1, y se marcan los candidatos del listado que caen dentro de una (columna `dentro_de_csl`), para que el equipo los revise.

El CSV debe tener `comment_id`, `thread_id`, `is_reply` y el texto normalizado con el mismo procedimiento que el piloto
(columna `text_normalized`; si no existe, se usa `comment_text` y se avisa).

In [ ]:
if APLICAR_CORPUS:
    ap = pd.read_csv(CORPUS_APLICACION)
    if "text_normalized" not in ap.columns:
        print("ATENCIÓN: no hay text_normalized; se usa comment_text sin normalizar (la comparación queda sesgada)")
        ap["text_normalized"] = ap["comment_text"]
    if MUESTRA_APLICACION and len(ap) > MUESTRA_APLICACION:
        ap = ap.sample(MUESTRA_APLICACION, random_state=0)
    ap["anotacion"] = np.nan
    cand_ap, _ = construir_candidatos(ap[["comment_id", "thread_id", "is_reply", "text_normalized", "anotacion"]], rx, set())
    cand_ap = anotar_gramatica(cand_ap, nlp)
    mg = elegir_margen_m(M2, cand, admitidas, embedder, MAX_ABSTENCION, 0)
    m_todo = M2(admitidas, embedder, seed=0).fit(cand)
    p_ap, ab_ap, prob_ap, org_ap, rev_ap = m_todo.predict(cand_ap, mg, detalle=True)
    cand_ap = cand_ap.assign(M2=p_ap, p_M2=prob_ap, abstiene=ab_ap, origen_M2=org_ap, revisar_categoria=rev_ap)
    csl_ap = detectar_csl(ap, nlp, "text_normalized")
    cand_ap = marcar_dentro_de_csl(cand_ap, csl_ap)
    print(f"Candidatas a CSL: {len(csl_ap)} en {csl_ap.comment_id.nunique()} comentarios · por regla:")
    print(csl_ap.regla_csl.value_counts().to_string())
    print(f"Candidatos del listado dentro de una candidata a CSL (marcados, para revisar): {(cand_ap.dentro_de_csl != '').sum()}")
    csl_ap.drop(columns=["coincidencia", "contexto"]).to_csv("resultados/publicable/aplicacion_candidatas_csl.csv", index=False)
    os.makedirs("resultados/interno", exist_ok=True)
    csl_ap.to_csv("resultados/interno/aplicacion_candidatas_csl_con_contexto.csv", index=False)
    display(csl_ap.groupby("regla_csl").head(3)[["regla_csl", "coincidencia", "contexto"]])
    ab_piloto = res[res.sistema == "M2 con abstención"].abstencion.mean()
    print(f"Comentarios: {len(ap)} · candidatos: {len(cand_ap)}")
    print(f"Abstención: corpus de aplicación {ab_ap.mean():.1%} · test del piloto {ab_piloto:.1%}")
    dist = pd.DataFrame({
        "aplicación": cand_ap[~cand_ap.abstiene].M2.value_counts(normalize=True),
        "piloto (test, M2)": todas.M2.value_counts(normalize=True),
    }).fillna(0).round(3)
    display(dist)
    cand_ap[["comment_id", "unidad", "inicio", "M2", "p_M2", "abstiene", "origen_M2", "revisar_categoria", "dentro_de_csl"]].to_csv(
        "resultados/publicable/aplicacion_M2.csv", index=False)
else:
    print("Sección desactivada (APLICAR_CORPUS = False)")

## 11. Guardar resultados

- `resultados/publicable/`: sin texto de comentarios; puede ir al repositorio.
- `resultados/interno/`: con ventanas de contexto; **no publicar** (políticas de la API de YouTube).

In [ ]:
os.makedirs("resultados/interno", exist_ok=True)
res.to_csv("resultados/publicable/resultados_semillas.csv", index=False)
tabla.to_csv("resultados/publicable/resumen_sistemas.csv")
ics.to_csv("resultados/publicable/intervalos_confianza.csv", index=False)
sin_texto = ["comment_id", "thread_id", "semilla", "unidad", "inicio", "oro", "B0", "B1", "M1", "M2",
             "p_M1", "p_M2", "origen_M2", "revisar_categoria", "abstiene_M2"]
todas[sin_texto].to_csv("resultados/publicable/predicciones_test.csv", index=False)
todas.drop(columns=["texto"]).to_csv("resultados/interno/predicciones_test_con_ventana.csv", index=False)
err.to_csv("resultados/interno/errores_semilla0.csv", index=False)
dentro_csl.to_csv("resultados/publicable/excluidas_dentro_de_csl.csv", index=False)
csl_piloto.drop(columns=["coincidencia", "contexto"]).to_csv("resultados/publicable/piloto_candidatas_csl.csv", index=False)
csl_piloto.to_csv("resultados/interno/piloto_candidatas_csl_con_contexto.csv", index=False)
print("Guardado en resultados/")